# Update the notebook templates

Clones the DataEngineering repo on this instance (first run) or updates it (later runs), then copies the
latest SageMaker notebook templates into `TEMPLATES_DIR`. Run it when you start work to pick up new or
changed templates.

**Before running:** `git` is available and GitHub is reachable from this instance. Nothing is written to
S3; the only changes are the clone and the copied templates on this instance.

**Stop the notebook instance when you finish** — SageMaker bills while it is running.

## Parameters

- `REPO_URL` — the repository to clone.
- `BRANCH` — branch to follow. Use `main`; set a pushed ticket branch to try templates before they merge.
- `CLONE_DIR` — where the repo is cloned. It is reset to `BRANCH` on every run, so don't edit files in it.
- `TEMPLATES_DIR` — where the latest templates are copied. Files here are overwritten on every run.
- `REPO_TEMPLATES_PATH` — the templates folder inside the repo.

In [ ]:
REPO_URL = "https://github.com/NMDSdevopsServiceAdm/DataEngineering.git"
BRANCH = "main"
CLONE_DIR = "~/SageMaker/DataEngineering"
TEMPLATES_DIR = "~/SageMaker/templates"
REPO_TEMPLATES_PATH = "notebooks/sagemaker"

## Setup

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

## Clone or update the repo

First run: a shallow clone of `BRANCH`. Later runs: fetch `BRANCH` and reset to it, so the clone always
matches GitHub even after a force-push. If git fails, its message is shown — usually GitHub access or the
branch name (it must exist and be pushed). Git never waits for a password and gives up after 5 minutes.

In [ ]:
clone_dir = Path(CLONE_DIR).expanduser()


def git(*args: str, cwd: "Path | None" = None) -> str:
    """Run a git command and return its output; raise with git's message if it fails."""
    result = subprocess.run(
        ["git", *args],
        cwd=cwd,
        capture_output=True,
        text=True,
        timeout=300,
        env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},  # fail instead of waiting for a password
    )
    if result.returncode:
        raise RuntimeError(f"git {' '.join(args)} failed:\n{result.stderr}")
    return result.stdout.strip()


if (clone_dir / ".git").exists():
    git("fetch", "--depth", "1", "origin", BRANCH, cwd=clone_dir)
    git("reset", "--hard", "FETCH_HEAD", cwd=clone_dir)
else:
    clone_dir.parent.mkdir(parents=True, exist_ok=True)
    git("clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(clone_dir))

print(f"{BRANCH} @ {git('log', '-1', '--format=%h %s', cwd=clone_dir)}")

## Copy the latest templates

Copies every template from the clone into `TEMPLATES_DIR`, overwriting the previous copies, and lists them.

In [ ]:
templates_dir = Path(TEMPLATES_DIR).expanduser()
templates_dir.mkdir(parents=True, exist_ok=True)

for template in sorted((clone_dir / REPO_TEMPLATES_PATH).glob("*.ipynb")):
    shutil.copy2(template, templates_dir / template.name)
    print(template.name)

## Using the templates

Open a template from `TEMPLATES_DIR` in the file browser, then **File → Save Notebook As…** into your own
folder before editing. Change the parameters cell only. Don't work inside `TEMPLATES_DIR` or `CLONE_DIR`:
both are overwritten on every run. To update this notebook itself, open the fresh copy from `TEMPLATES_DIR`.